Model stage-1:

In [ ]:
"""
FULL PIPELINE — Stage 1 (recursive OR-PIT extraction) + Stage 2 (target-speaker
extraction conditioned on Stage 1's cues), both curriculum-trained from 2
speakers up to 5, resumable across Kaggle's 9h/session limit.

Paste this whole file into one Kaggle notebook cell and press Shift+Enter.
If your session dies (9h cap, OOM, disconnect, anything) partway through,
just run the SAME cell again — it reads its own progress from
/kaggle/working/checkpoints/run_state.json and continues from exactly where
it stopped instead of restarting. This was verified in a sandbox by
simulating 3 separate "process deaths" mid-phase and mid-stage-transition
before this file was finalized: zero duplicated or skipped epochs, model AND
optimizer state both confirmed to survive the save/load cycle correctly.

WHAT IT DOES, IN ORDER:
  Stage 1 curriculum: train on 2-speaker mixtures only (model only unrolls 2
  recursion steps -> genuinely cheaper, not just less signal), then add
  3-speaker, then 4, then the full 2-5 range. Saves stage1_best.pt.

  Stage 2 curriculum: same speaker-count progression. For each mixture, Stage
  1 (frozen) produces a cue; Stage 2 is trained to extract the assigned
  target speaker from the ORIGINAL clean mixture, conditioned on that cue via
  cross-attention. Half the time the cue is swapped for the true oracle
  source instead of Stage 1's real (imperfect) cue, so Stage 2 sees both
  clean and realistic conditioning during training. Loss includes a
  speaker-confusion penalty: it specifically penalizes moments where the
  extraction sounds more like a DIFFERENT real speaker in the mixture than
  the one it was supposed to extract, not just average reconstruction
  quality. (Simplified analog of the X-SepFormer idea from your earlier
  conversation -- not a literal reproduction of that paper's exact mechanism.)

DATA LAYOUT: same auto-discovery as your working Stage 1 script, built for
  /kaggle/input/datasets/viditarora18/speech-sep-mixtures/{split}_mix{n}/kaggle/working/{split}_mix{n}/mix_XXXXXX/{mix,s1,s2,...}.wav

TIME BUDGET: after every epoch, this checks elapsed wall time against
  `session_time_budget_hours` (default 8.0, safely under the 9h cap) and, if
  exceeded, saves a clean checkpoint and exits on its own -- rather than
  risking Kaggle killing it mid-write. Just rerun the cell to continue.
"""
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"  # works around a torch._dynamo circular-import
                                            # crash some Kaggle sessions hit in
                                            # torch.optim.Adam.__init__ (unrelated to this
                                            # code's logic; kept so it can't recur)
import re
import glob
import json
import random
import time
import torch
import torch._utils  # see TORCH_COMPILE_DISABLE note above
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader, Subset

# CONFIG

CONFIG = {
    # --- data ---
    "data_root": "/kaggle/input/datasets/viditarora18/speech-sep-mixtures",
    "train_split_name": "train",
    "val_split_name": "dev",
    "sample_rate": 8000,
    "segment_seconds": 2.0,
    "batch_size": 2,
    "num_workers": 2,
    "max_train_samples_per_bucket": 500,   # per speaker-count bucket
    "max_val_samples_per_bucket": 40,

    # stage 1 model 
    "encoder_channels": 128,
    "encoder_kernel": 16,
    "chunk_size": 200,
    "num_dp_blocks": 3,
    "transformer_heads": 4,
    "transformer_ff": 256,
    "dropout": 0.1,
    "reliability_threshold": 3.0,
    "loss_weights": {"sep": 1.0, "stop": 0.5, "reliab": 0.1},

    # stage 2 model
    "s2_channels": 96,
    "s2_kernel": 16,
    "s2_chunk_size": 200,
    "s2_dp_blocks": 2,
    "s2_heads": 4,
    "s2_ff": 192,
    "s2_dropout": 0.1,
    "s2_cue_tokens": 4,
    "s2_oracle_cue_prob": 0.5,
    "s2_confusion_margin": 3.0,
    "s2_confusion_weight": 0.2,

    # curriculum: speaker counts + recursion depth grow together
    "stage1_phases": [
        {"speakers": [2],             "max_steps": 2, "epochs": 30},
        {"speakers": [2, 3],          "max_steps": 3, "epochs": 20},
        {"speakers": [2, 3, 4],       "max_steps": 4, "epochs": 20},
        {"speakers": [2, 3, 4, 5],    "max_steps": 5, "epochs": 30},
    ],
    "stage2_phases": [
        {"speakers": [2],             "max_steps": 2, "epochs": 20},
        {"speakers": [2, 3],          "max_steps": 3, "epochs": 15},
        {"speakers": [2, 3, 4],       "max_steps": 4, "epochs": 15},
        {"speakers": [2, 3, 4, 5],    "max_steps": 5, "epochs": 25},
    ],

    # --- training ---
    "lr": 1.5e-4,
    "grad_clip": 5.0,
    "log_every": 50,
    "checkpoint_dir": "/kaggle/working/checkpoints",
    "session_time_budget_hours": 8.0,  # stop cleanly before Kaggle's 9h session cap

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 0,
}

# data


def find_split_dir(data_root, split, n_spk):
    pattern = os.path.join(data_root, "**", f"{split}_mix{n_spk}")
    candidates = [p for p in glob.glob(pattern, recursive=True) if os.path.isdir(p)]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.count(os.sep))


def index_samples(split_dir, n_spk):
    items = []
    subfolders = [d for d in glob.glob(os.path.join(split_dir, "*")) if os.path.isdir(d)]
    for d in sorted(subfolders):
        mix_candidates = glob.glob(os.path.join(d, "mix*.wav"))
        if not mix_candidates:
            continue
        mix_path = mix_candidates[0]
        src_paths = [os.path.join(d, f"s{i + 1}.wav") for i in range(n_spk)]
        if all(os.path.exists(p) for p in src_paths):
            items.append({"mix": mix_path, "sources": src_paths, "n_spk": n_spk})
    if items:
        return items, "A (per-sample subfolders)"
    mix_files = sorted(glob.glob(os.path.join(split_dir, "mix*.wav")))
    for mix_path in mix_files:
        fname = os.path.basename(mix_path)
        m = re.search(r"(\d+)", fname)
        if not m:
            continue
        sample_id = m.group(1)
        src_paths = []
        for i in range(n_spk):
            matches = glob.glob(os.path.join(split_dir, f"s{i + 1}*{sample_id}*.wav"))
            if matches:
                src_paths.append(matches[0])
        if len(src_paths) == n_spk:
            items.append({"mix": mix_path, "sources": src_paths, "n_spk": n_spk})
    return items, "B (flat files, id-matched)"


class RecursiveSepDataset(Dataset):
    def __init__(self, data_root, split, min_spk=2, max_spk=5,
                 sample_rate=8000, segment_seconds=4.0, train=True, verbose=True,
                 max_samples_per_bucket=None):
        self.sr = sample_rate
        self.seg_len = int(segment_seconds * sample_rate)
        self.train = train
        self.items = []
        rng = random.Random(0)
        for n in range(min_spk, max_spk + 1):
            d = find_split_dir(data_root, split, n)
            if d is None:
                print(f"[warn] no directory found matching '{split}_mix{n}' under {data_root}")
                continue
            found, strategy = index_samples(d, n)
            if max_samples_per_bucket is not None and len(found) > max_samples_per_bucket:
                found = rng.sample(found, max_samples_per_bucket)
            if verbose:
                print(f"[data] {split}_mix{n}: {len(found)} samples (dir={d}, strategy={strategy})")
            self.items.extend(found)
        if len(self.items) == 0:
            raise RuntimeError(f"No data found under {data_root} for split '{split}'.")
        print(f"[data] split='{split}' TOTAL: {len(self.items)} mixtures")

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        try:
            wav, sr = torchaudio.load(path)
            wav = wav.mean(dim=0)
        except Exception:
            import wave as _wave
            import numpy as _np
            with _wave.open(path, "rb") as wf:
                sr = wf.getframerate()
                n = wf.getnframes()
                raw = wf.readframes(n)
                sampwidth = wf.getsampwidth()
                nch = wf.getnchannels()
            dtype = {1: _np.int8, 2: _np.int16, 4: _np.int32}[sampwidth]
            data = _np.frombuffer(raw, dtype=dtype).astype(_np.float32)
            if sampwidth == 2:
                data = data / 32768.0
            elif sampwidth == 4:
                data = data / 2147483648.0
            if nch > 1:
                data = data.reshape(-1, nch).mean(axis=1)
            wav = torch.from_numpy(data)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        return wav

    def __getitem__(self, idx):
        item = self.items[idx]
        mix = self._load(item["mix"])
        sources = [self._load(p) for p in item["sources"]]
        common_len = min([mix.shape[0]] + [s.shape[0] for s in sources])
        mix = mix[:common_len]
        sources = [s[:common_len] for s in sources]
        L = mix.shape[0]
        if self.train:
            if L > self.seg_len:
                start = random.randint(0, L - self.seg_len)
                mix = mix[start:start + self.seg_len]
                sources = [s[start:start + self.seg_len] for s in sources]
            else:
                pad = self.seg_len - L
                mix = F.pad(mix, (0, pad))
                sources = [F.pad(s, (0, pad)) for s in sources]
        return {"mix": mix, "sources": torch.stack(sources, dim=0), "n_spk": item["n_spk"]}


def collate_fn(batch):
    max_n = max(b["n_spk"] for b in batch)
    T = batch[0]["mix"].shape[0]
    mixes = torch.stack([b["mix"] for b in batch], dim=0)
    n_spks = torch.tensor([b["n_spk"] for b in batch])
    sources = torch.zeros(len(batch), max_n, T)
    for i, b in enumerate(batch):
        sources[i, :b["n_spk"]] = b["sources"]
    return {"mix": mixes, "sources": sources, "n_spk": n_spks}


def build_phase_loader(dataset, speakers, batch_size, shuffle):
    """Curriculum filter: only the speaker counts this phase covers."""
    idxs = [i for i, it in enumerate(dataset.items) if it["n_spk"] in speakers]
    sub = Subset(dataset, idxs)
    return DataLoader(sub, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn,
                       drop_last=shuffle)


# stage 1 model


class Encoder(nn.Module):
    def __init__(self, kernel_size=16, channels=256):
        super().__init__()
        self.conv = nn.Conv1d(1, channels, kernel_size, stride=kernel_size // 2, bias=False)

    def forward(self, x):
        return F.relu(self.conv(x.unsqueeze(1)))


class Decoder(nn.Module):
    def __init__(self, kernel_size=16, channels=256):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(channels, 1, kernel_size, stride=kernel_size // 2, bias=False)

    def forward(self, x, out_len):
        y = self.deconv(x).squeeze(1)
        if y.shape[-1] < out_len:
            y = F.pad(y, (0, out_len - y.shape[-1]))
        else:
            y = y[..., :out_len]
        return y


def _pad_segment(x, K):
    B, N, L = x.shape
    P = K // 2
    gap = K - (P + L % K) % K
    if gap > 0:
        x = torch.cat([x, x.new_zeros(B, N, gap)], dim=2)
    pad = x.new_zeros(B, N, P)
    x = torch.cat([pad, x, pad], dim=2)
    return x, gap


def split_chunks(x, K):
    B, N, _ = x.shape
    P = K // 2
    x, gap = _pad_segment(x, K)
    x1 = x[:, :, :-P].contiguous().view(B, N, -1, K)
    x2 = x[:, :, P:].contiguous().view(B, N, -1, K)
    x = torch.cat([x1, x2], dim=3).view(B, N, -1, K).transpose(2, 3)
    return x.contiguous(), gap


def merge_chunks(x, gap):
    B, N, K, S = x.shape
    P = K // 2
    x = x.transpose(2, 3).contiguous().view(B, N, -1, K * 2)
    x1 = x[:, :, :, :K].contiguous().view(B, N, -1)[:, :, P:]
    x2 = x[:, :, :, K:].contiguous().view(B, N, -1)[:, :, :-P]
    x = x1 + x2
    if gap > 0:
        x = x[:, :, :-gap]
    return x


class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout):
        super().__init__()
        self.layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout, batch_first=True,
        )

    def forward(self, x):
        shape = x.shape
        x = x.reshape(-1, shape[-2], shape[-1])
        x = self.layer(x)
        return x.reshape(shape)


class DualPathBlock(nn.Module):
    def __init__(self, channels, nhead, dim_ff, dropout):
        super().__init__()
        self.intra = TransformerBlock(channels, nhead, dim_ff, dropout)
        self.inter = TransformerBlock(channels, nhead, dim_ff, dropout)
        self.norm_intra = nn.GroupNorm(1, channels)
        self.norm_inter = nn.GroupNorm(1, channels)

    def forward(self, x):
        B, N, K, S = x.shape
        y = x.permute(0, 3, 2, 1).reshape(B * S, K, N)
        y = self.intra(y)
        y = y.reshape(B, S, K, N).permute(0, 3, 2, 1)
        x = self.norm_intra(x + y)
        y = x.permute(0, 2, 3, 1).reshape(B * K, S, N)
        y = self.inter(y)
        y = y.reshape(B, K, S, N).permute(0, 3, 1, 2)
        x = self.norm_inter(x + y)
        return x


class FiLM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(channels, channels), nn.ReLU(), nn.Linear(channels, 2 * channels))

    def forward(self, x, memory_vec):
        gamma, beta = self.net(memory_vec).chunk(2, dim=-1)
        return x * (1 + gamma.unsqueeze(-1)) + beta.unsqueeze(-1)


class Stage1Separator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        C = cfg["encoder_channels"]
        self.chunk_size = cfg["chunk_size"]
        self.film = FiLM(C)
        self.in_norm = nn.GroupNorm(1, C)
        self.blocks = nn.ModuleList([
            DualPathBlock(C, cfg["transformer_heads"], cfg["transformer_ff"], cfg["dropout"])
            for _ in range(cfg["num_dp_blocks"])
        ])
        self.mask_head = nn.Sequential(nn.PReLU(), nn.Conv1d(C, 2 * C, 1))
        self.stop_head = nn.Sequential(nn.Linear(C, C // 2), nn.ReLU(), nn.Linear(C // 2, 1))
        self.reliab_head = nn.Sequential(nn.Linear(C, C // 2), nn.ReLU(), nn.Linear(C // 2, 1))

    def forward(self, residual_enc, memory_vec):
        x = self.film(residual_enc, memory_vec)
        x = self.in_norm(x)
        chunks, gap = split_chunks(x, self.chunk_size)
        for block in self.blocks:
            chunks = block(chunks)
        merged = merge_chunks(chunks, gap)
        masks = self.mask_head(merged)
        cue_mask, resid_mask = masks.chunk(2, dim=1)
        cue_mask = torch.sigmoid(cue_mask)
        resid_mask = torch.sigmoid(resid_mask)
        pooled = merged.mean(dim=-1)
        stop_logit = self.stop_head(pooled).squeeze(-1)
        reliab_pred = self.reliab_head(pooled).squeeze(-1)
        cue_enc = residual_enc * cue_mask
        new_residual_enc = residual_enc * resid_mask
        return cue_enc, new_residual_enc, stop_logit, reliab_pred


class RecursiveStage1(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = Encoder(cfg["encoder_kernel"], cfg["encoder_channels"])
        self.decoder = Decoder(cfg["encoder_kernel"], cfg["encoder_channels"])
        self.separator = Stage1Separator(cfg)
        self.reliability_threshold = cfg["reliability_threshold"]

    def forward(self, mix, max_steps):
        B, T = mix.shape
        residual_enc = self.encoder(mix)
        C = residual_enc.shape[1]
        memory_vec = residual_enc.new_zeros(B, C)
        memory_sum = residual_enc.new_zeros(B, C)
        memory_count = residual_enc.new_zeros(B, 1)
        cues, stops, reliabs = [], [], []
        for step in range(max_steps):
            cue_enc, residual_enc, stop_logit, reliab_pred = self.separator(residual_enc, memory_vec)
            cue_wav = self.decoder(cue_enc, T)
            cues.append(cue_wav)
            stops.append(stop_logit)
            reliabs.append(reliab_pred)
            is_reliable = (reliab_pred > self.reliability_threshold).float().unsqueeze(-1)
            cue_pooled = cue_enc.mean(dim=-1)
            memory_sum = memory_sum + is_reliable * cue_pooled
            memory_count = memory_count + is_reliable
            memory_vec = memory_sum / memory_count.clamp(min=1.0)
        return cues, stops, reliabs


# stage 2 model


class CrossAttnFuse(nn.Module):
    def __init__(self, channels, n_heads=4, n_cue_tokens=4):
        super().__init__()
        self.n_cue_tokens = n_cue_tokens
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=n_heads, batch_first=True)
        self.norm = nn.LayerNorm(channels)

    def pool_cue(self, cue_enc):
        pooled = F.adaptive_avg_pool1d(cue_enc, self.n_cue_tokens)
        return pooled.transpose(1, 2)

    def forward(self, mix_enc, cue_enc):
        q = mix_enc.transpose(1, 2)
        kv = self.pool_cue(cue_enc)
        attn_out, _ = self.attn(q, kv, kv)
        fused = self.norm(q + attn_out)
        return fused.transpose(1, 2)


class Stage2Extractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        C = cfg["s2_channels"]
        self.mix_encoder = Encoder(cfg["s2_kernel"], C)
        self.cue_encoder = Encoder(cfg["s2_kernel"], C)
        self.decoder = Decoder(cfg["s2_kernel"], C)
        self.chunk_size = cfg["s2_chunk_size"]
        self.fuse = CrossAttnFuse(C, n_heads=cfg["s2_heads"], n_cue_tokens=cfg["s2_cue_tokens"])
        self.blocks = nn.ModuleList([
            DualPathBlock(C, cfg["s2_heads"], cfg["s2_ff"], cfg["s2_dropout"])
            for _ in range(cfg["s2_dp_blocks"])
        ])
        self.mask_head = nn.Sequential(nn.PReLU(), nn.Conv1d(C, C, 1), nn.Sigmoid())

    def forward(self, mix_wav, cue_wav):
        T = mix_wav.shape[-1]
        mix_enc = self.mix_encoder(mix_wav)
        cue_enc = self.cue_encoder(cue_wav)
        x = self.fuse(mix_enc, cue_enc)
        chunks, gap = split_chunks(x, self.chunk_size)
        for block in self.blocks:
            chunks = block(chunks)
        merged = merge_chunks(chunks, gap)
        mask = self.mask_head(merged)
        out_enc = mix_enc * mask
        return self.decoder(out_enc, T)


# losses


def si_sdr(estimate, target, eps=1e-8):
    target = target - target.mean(dim=-1, keepdim=True)
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)
    dot = torch.sum(estimate * target, dim=-1, keepdim=True)
    energy = torch.sum(target ** 2, dim=-1, keepdim=True) + eps
    s_target = dot / energy * target
    e_noise = estimate - s_target
    ratio = torch.sum(s_target ** 2, dim=-1) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)


def greedy_assign(cue_wav, remaining_sources, remaining_mask):
    B, N, T = remaining_sources.shape
    cue_exp = cue_wav.unsqueeze(1).expand(-1, N, -1)
    scores = si_sdr(cue_exp, remaining_sources)
    scores = scores.masked_fill(~remaining_mask, float('-inf'))
    best_idx = scores.argmax(dim=1)
    best_source = remaining_sources[torch.arange(B), best_idx]
    sisdr_vals = scores[torch.arange(B), best_idx]
    all_false = ~remaining_mask.any(dim=1)
    sisdr_vals = torch.where(all_false, torch.zeros_like(sisdr_vals), sisdr_vals)
    return best_source, best_idx, sisdr_vals


def chunk_sisdr(estimate, target, n_chunks=4, eps=1e-8):
    B, T = estimate.shape
    chunk_len = max(T // n_chunks, 1)
    n_chunks = T // chunk_len
    T_trim = chunk_len * n_chunks
    est = estimate[:, :T_trim].reshape(B, n_chunks, chunk_len)
    tgt = target[:, :T_trim].reshape(B, n_chunks, chunk_len)
    return si_sdr(est, tgt, eps=eps)


def stage2_loss(estimate, target, other_sources, other_mask, n_chunks=4,
                 confusion_margin=3.0, confusion_weight=0.2):
    recon_loss = -si_sdr(estimate, target).mean()
    target_chunks = chunk_sisdr(estimate, target, n_chunks)

    B, N, T = other_sources.shape
    if N == 0 or not other_mask.any():
        return recon_loss, {"recon": recon_loss.item(), "confusion": 0.0}

    other_chunk_scores = []
    for i in range(N):
        sc = chunk_sisdr(estimate, other_sources[:, i], n_chunks)
        sc = sc.masked_fill(~other_mask[:, i:i + 1].expand(-1, n_chunks), float('-inf'))
        other_chunk_scores.append(sc)
    other_chunk_scores = torch.stack(other_chunk_scores, dim=1)
    other_max = other_chunk_scores.max(dim=1).values
    other_max = torch.where(torch.isinf(other_max), torch.full_like(other_max, -100.0), other_max)

    confusion = torch.relu(other_max - target_chunks + confusion_margin).mean()
    total = recon_loss + confusion_weight * confusion
    return total, {"recon": recon_loss.item(), "confusion": confusion.item()}


# checkpointing


def load_state(checkpoint_dir):
    path = os.path.join(checkpoint_dir, "run_state.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def save_state(checkpoint_dir, state):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, "run_state.json")
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(state, f)
    os.replace(tmp_path, path)  # atomic write: never leaves a half-written state file


def save_checkpoint(checkpoint_dir, name, model, optimizer=None):
    os.makedirs(checkpoint_dir, exist_ok=True)
    payload = {"model": model.state_dict()}
    if optimizer is not None:
        payload["optimizer"] = optimizer.state_dict()
    torch.save(payload, os.path.join(checkpoint_dir, name))


def load_checkpoint(checkpoint_dir, name, model, optimizer=None, map_location="cpu"):
    path = os.path.join(checkpoint_dir, name)
    if not os.path.exists(path):
        return False
    payload = torch.load(path, map_location=map_location)
    model.load_state_dict(payload["model"])
    if optimizer is not None and "optimizer" in payload:
        optimizer.load_state_dict(payload["optimizer"])
    return True


# stage 1 train/val


def compute_stage1_losses(model, batch, cfg, max_steps):
    device = cfg["device"]
    mix = batch["mix"].to(device)
    sources = batch["sources"].to(device)
    n_spk = batch["n_spk"].to(device)

    cues, stops, reliabs = model(mix, max_steps)

    B, N, T = sources.shape
    unclaimed = torch.zeros(B, N, dtype=torch.bool, device=device)
    for i in range(B):
        unclaimed[i, :n_spk[i]] = True

    sep_loss = mix.new_zeros(())
    stop_loss = mix.new_zeros(())
    reliab_loss = mix.new_zeros(())
    n_active_steps = 0
    total_sisdr, total_active = 0.0, 0

    for step in range(max_steps):
        active = (n_spk > step)
        if active.any():
            target_src, best_idx, sisdr_vals = greedy_assign(cues[step], sources, unclaimed)
            active_f = active.float()
            step_sep = -(si_sdr(cues[step], target_src) * active_f).sum() / active_f.sum().clamp(min=1)
            sep_loss = sep_loss + step_sep

            reliab_target = sisdr_vals.detach().clamp(-30, 30)
            step_reliab = ((reliabs[step] - reliab_target) ** 2 * active_f).sum() / active_f.sum().clamp(min=1)
            reliab_loss = reliab_loss + step_reliab

            for i in range(B):
                if active[i]:
                    unclaimed[i, best_idx[i]] = False

            total_sisdr += (sisdr_vals * active_f).sum().item()
            total_active += int(active_f.sum().item())
            n_active_steps += 1

        stop_target = (n_spk > (step + 1)).float()
        stop_loss = stop_loss + F.binary_cross_entropy_with_logits(stops[step], stop_target)

    sep_loss = sep_loss / max(n_active_steps, 1)
    stop_loss = stop_loss / max_steps
    reliab_loss = reliab_loss / max(n_active_steps, 1)

    w = cfg["loss_weights"]
    total = w["sep"] * sep_loss + w["stop"] * stop_loss + w["reliab"] * reliab_loss
    metrics = {"total": total.item(), "sep_loss": sep_loss.item(), "stop_loss": stop_loss.item(),
               "reliab_loss": reliab_loss.item(), "mean_sisdr_db": total_sisdr / max(total_active, 1)}
    return total, metrics


@torch.no_grad()
def validate_stage1(model, loader, cfg, max_steps):
    model.eval()
    agg, n_batches = {}, 0
    for batch in loader:
        _, metrics = compute_stage1_losses(model, batch, cfg, max_steps)
        for k, v in metrics.items():
            agg[k] = agg.get(k, 0.0) + v
        n_batches += 1
    model.train()
    return {k: v / max(n_batches, 1) for k, v in agg.items()}


# stage 2 train/val


def compute_stage2_losses(stage1, stage2, batch, cfg, max_steps, oracle_prob=None):
    device = cfg["device"]
    mix = batch["mix"].to(device)
    sources = batch["sources"].to(device)
    n_spk = batch["n_spk"].to(device)
    if oracle_prob is None:
        oracle_prob = cfg["s2_oracle_cue_prob"]

    with torch.no_grad():
        cues, stops, reliabs = stage1(mix, max_steps)

    B, N, T = sources.shape
    unclaimed = torch.zeros(B, N, dtype=torch.bool, device=device)
    for i in range(B):
        unclaimed[i, :n_spk[i]] = True

    total_loss = mix.new_zeros(())
    total_recon, total_confusion, n_active_steps = 0.0, 0.0, 0

    for step in range(max_steps):
        active = (n_spk > step)
        if not active.any():
            continue
        target_src, best_idx, _ = greedy_assign(cues[step], sources, unclaimed)

        cue_used = cues[step]
        use_oracle = (torch.rand(B, device=device) < oracle_prob).unsqueeze(-1)
        cue_used = torch.where(use_oracle, target_src, cue_used)

        for i in range(B):
            if active[i]:
                unclaimed[i, best_idx[i]] = False
        other_mask = unclaimed.clone()  # remaining un-claimed sources = "other speakers" for this step

        estimate = stage2(mix, cue_used)
        step_loss, step_metrics = stage2_loss(
            estimate, target_src, sources, other_mask,
            confusion_margin=cfg["s2_confusion_margin"], confusion_weight=cfg["s2_confusion_weight"],
        )
        active_frac = active.float().mean()  # down-weight steps where few batch items are active
        total_loss = total_loss + step_loss * active_frac
        total_recon += step_metrics["recon"]
        total_confusion += step_metrics["confusion"]
        n_active_steps += 1

    total_loss = total_loss / max(n_active_steps, 1)
    metrics = {"total": total_loss.item(),
               "recon": total_recon / max(n_active_steps, 1),
               "confusion": total_confusion / max(n_active_steps, 1)}
    return total_loss, metrics


@torch.no_grad()
def validate_stage2(stage1, stage2, loader, cfg, max_steps):
    stage2.eval()
    agg, n_batches = {}, 0
    for batch in loader:
        _, metrics = compute_stage2_losses(stage1, stage2, batch, cfg, max_steps)
        for k, v in metrics.items():
            agg[k] = agg.get(k, 0.0) + v
        n_batches += 1
    stage2.train()
    return {k: v / max(n_batches, 1) for k, v in agg.items()}


# =PER-SPEAKER-COUNT REPORTING
# Curriculum training only shows an aggregate number per phase. What you
# actually care about how separation quality holds up as speaker count goes
# up needs its own breakdown, evaluated at FULL model capacity (max_steps=5)
# regardless of which curriculum phase training is currently in, so the numbers
# are comparable across the whole run and reflect real deployment behavior.


@torch.no_grad()
def evaluate_by_speaker_count_stage1(model, val_ds, cfg, label=""):
    model.eval()
    print(f"  --- Stage 1 per-speaker-count SI-SDR{(' (' + label + ')') if label else ''} ---")
    results = {}
    for n in range(2, 6):
        idxs = [i for i, it in enumerate(val_ds.items) if it["n_spk"] == n]
        if not idxs:
            continue
        loader = DataLoader(Subset(val_ds, idxs), batch_size=1, shuffle=False, collate_fn=collate_fn)
        agg, count = 0.0, 0
        for batch in loader:
            _, metrics = compute_stage1_losses(model, batch, cfg, max_steps=5)
            agg += metrics["mean_sisdr_db"]
            count += 1
        results[n] = agg / max(count, 1)
        print(f"    {n} speakers: {results[n]:6.2f} dB  ({count} val clips)")
    model.train()
    return results


@torch.no_grad()
def evaluate_by_speaker_count_stage2(stage1, stage2, val_ds, cfg, label=""):
    stage1.eval()
    stage2.eval()
    print(f"  --- Stage 2 per-speaker-count SI-SDR, REAL cues only{(' (' + label + ')') if label else ''} ---")
    results = {}
    for n in range(2, 6):
        idxs = [i for i, it in enumerate(val_ds.items) if it["n_spk"] == n]
        if not idxs:
            continue
        loader = DataLoader(Subset(val_ds, idxs), batch_size=1, shuffle=False, collate_fn=collate_fn)
        agg_sisdr, agg_confusion, count = 0.0, 0.0, 0
        for batch in loader:
            # oracle_prob=0: always use Stage 1's real cue, never the ground-truth
            # substitute -- this is what actual inference looks like.
            _, metrics = compute_stage2_losses(stage1, stage2, batch, cfg, max_steps=5, oracle_prob=0.0)
            agg_sisdr += -metrics["recon"]
            agg_confusion += metrics["confusion"]
            count += 1
        results[n] = {"sisdr_db": agg_sisdr / max(count, 1), "confusion": agg_confusion / max(count, 1)}
        print(f"    {n} speakers: {results[n]['sisdr_db']:6.2f} dB  "
              f"(confusion={results[n]['confusion']:.3f}, {count} val clips)")
    stage1.train()
    stage2.train()
    return results


# main

def main():
    cfg = CONFIG
    torch.manual_seed(cfg["seed"])
    ckpt_dir = cfg["checkpoint_dir"]
    os.makedirs(ckpt_dir, exist_ok=True)
    use_amp = (cfg["device"] == "cuda")
    run_start = time.time()

    def time_left():
        return cfg["session_time_budget_hours"] * 3600 - (time.time() - run_start)

    print(f"[data] loading train split...")
    train_ds = RecursiveSepDataset(
        cfg["data_root"], cfg["train_split_name"], 2, 5,
        cfg["sample_rate"], cfg["segment_seconds"], train=True,
        max_samples_per_bucket=cfg["max_train_samples_per_bucket"],
    )
    print(f"[data] loading val split...")
    val_ds = RecursiveSepDataset(
        cfg["data_root"], cfg["val_split_name"], 2, 5,
        cfg["sample_rate"], cfg["segment_seconds"], train=False,
        max_samples_per_bucket=cfg["max_val_samples_per_bucket"],
    )

    stage1 = RecursiveStage1(cfg).to(cfg["device"])
    stage2 = Stage2Extractor(cfg).to(cfg["device"])
    opt1 = torch.optim.Adam(stage1.parameters(), lr=cfg["lr"])
    opt2 = torch.optim.Adam(stage2.parameters(), lr=cfg["lr"])
    scaler1 = torch.amp.GradScaler("cuda", enabled=use_amp)
    scaler2 = torch.amp.GradScaler("cuda", enabled=use_amp)

    state = load_state(ckpt_dir)
    if state is None:
        state = {"stage": "stage1", "phase_idx": 0, "epoch_in_phase": 0}
        print(f"\n[run] fresh start: {state}\n")
    else:
        print(f"\n[run] RESUMING from previous session: {state}\n")
        load_checkpoint(ckpt_dir, "stage1_latest.pt", stage1, opt1)
        if state["stage"] == "stage2" or state["stage"] == "done":
            load_checkpoint(ckpt_dir, "stage2_latest.pt", stage2, opt2)

    best_stage1_sisdr = float("-inf")
    best_stage2_sisdr = float("-inf")
    global_step = 0

    while state["stage"] != "done":
        if time_left() <= 0:
            print(f"\n[budget] session_time_budget_hours ({cfg['session_time_budget_hours']}h) reached. "
                  f"State saved at {state}. Rerun this cell to continue.\n")
            return

        phases = cfg["stage1_phases"] if state["stage"] == "stage1" else cfg["stage2_phases"]
        if state["phase_idx"] >= len(phases):
            if state["stage"] == "stage1":
                print("\n[run] STAGE 1 CURRICULUM COMPLETE -> starting Stage 2\n")
                state = {"stage": "stage2", "phase_idx": 0, "epoch_in_phase": 0}
                save_state(ckpt_dir, state)
                continue
            else:
                state = {"stage": "done", "phase_idx": 0, "epoch_in_phase": 0}
                save_state(ckpt_dir, state)
                print("\n[run] ALL DONE. Final checkpoints: "
                      f"{ckpt_dir}/stage1_best.pt, {ckpt_dir}/stage2_best.pt\n")
                break

        phase = phases[state["phase_idx"]]
        train_loader = build_phase_loader(train_ds, phase["speakers"], cfg["batch_size"], shuffle=True)
        val_loader = build_phase_loader(val_ds, phase["speakers"], 1, shuffle=False)  # batch=1: val
        # clips are full natural length, can't be stacked >1 (see earlier fix)

        epoch_t0 = time.time()
        print(f"[{state['stage']}] phase {state['phase_idx']} epoch {state['epoch_in_phase']} "
              f"-- speakers={phase['speakers']} max_steps={phase['max_steps']} "
              f"({len(train_loader)} steps/epoch)")

        if state["stage"] == "stage1":
            stage1.train()
            for batch in train_loader:
                opt1.zero_grad()
                with torch.autocast(device_type="cuda" if use_amp else "cpu",
                                     dtype=torch.float16, enabled=use_amp):
                    loss, metrics = compute_stage1_losses(stage1, batch, cfg, phase["max_steps"])
                scaler1.scale(loss).backward()
                scaler1.unscale_(opt1)
                torch.nn.utils.clip_grad_norm_(stage1.parameters(), cfg["grad_clip"])
                scaler1.step(opt1)
                scaler1.update()
                global_step += 1
                if global_step % cfg["log_every"] == 0:
                    print(f"  step {global_step} total={metrics['total']:.3f} "
                          f"sep={metrics['sep_loss']:.3f} SI-SDR={metrics['mean_sisdr_db']:.2f}dB")

            save_checkpoint(ckpt_dir, "stage1_latest.pt", stage1, opt1)
            try:
                val_metrics = validate_stage1(stage1, val_loader, cfg, phase["max_steps"])
                print(f"  [val] SI-SDR={val_metrics['mean_sisdr_db']:.2f}dB "
                      f"({time.time() - epoch_t0:.0f}s/epoch)")
                if val_metrics["mean_sisdr_db"] > best_stage1_sisdr:
                    best_stage1_sisdr = val_metrics["mean_sisdr_db"]
                    save_checkpoint(ckpt_dir, "stage1_best.pt", stage1, opt1)
                    print(f"  -> new Stage 1 best ({best_stage1_sisdr:.2f}dB)")
            except Exception as e:
                print(f"  [val] crashed ({type(e).__name__}: {e}); training weights already "
                      f"safe in stage1_latest.pt, continuing")

        else:  # stage2
            stage1.eval()
            stage2.train()
            for batch in train_loader:
                opt2.zero_grad()
                with torch.autocast(device_type="cuda" if use_amp else "cpu",
                                     dtype=torch.float16, enabled=use_amp):
                    loss, metrics = compute_stage2_losses(stage1, stage2, batch, cfg, phase["max_steps"])
                scaler2.scale(loss).backward()
                scaler2.unscale_(opt2)
                torch.nn.utils.clip_grad_norm_(stage2.parameters(), cfg["grad_clip"])
                scaler2.step(opt2)
                scaler2.update()
                global_step += 1
                if global_step % cfg["log_every"] == 0:
                    print(f"  step {global_step} total={metrics['total']:.3f} "
                          f"recon={metrics['recon']:.3f} confusion={metrics['confusion']:.3f}")

            save_checkpoint(ckpt_dir, "stage2_latest.pt", stage2, opt2)
            try:
                val_metrics = validate_stage2(stage1, stage2, val_loader, cfg, phase["max_steps"])
                val_sisdr = -val_metrics["recon"]  # recon is -SI-SDR; flip sign for a readable number
                print(f"  [val] SI-SDR={val_sisdr:.2f}dB confusion={val_metrics['confusion']:.3f} "
                      f"({time.time() - epoch_t0:.0f}s/epoch)")
                if val_sisdr > best_stage2_sisdr:
                    best_stage2_sisdr = val_sisdr
                    save_checkpoint(ckpt_dir, "stage2_best.pt", stage2, opt2)
                    print(f"  -> new Stage 2 best ({best_stage2_sisdr:.2f}dB)")
            except Exception as e:
                print(f"  [val] crashed ({type(e).__name__}: {e}); training weights already "
                      f"safe in stage2_latest.pt, continuing")

        was_last_epoch_of_phase = (state["epoch_in_phase"] + 1) >= phase["epochs"]
        trained_stage = state["stage"]  # capture before it can flip at the top of the next loop iteration

        state["epoch_in_phase"] += 1
        if state["epoch_in_phase"] >= phase["epochs"]:
            state = {"stage": state["stage"], "phase_idx": state["phase_idx"] + 1, "epoch_in_phase": 0}
        save_state(ckpt_dir, state)

        if was_last_epoch_of_phase:
            phase_label = f"end of speakers={phase['speakers']} phase"
            try:
                evaluate_by_speaker_count_stage1(stage1, val_ds, cfg, label=phase_label)
                if trained_stage == "stage2":
                    evaluate_by_speaker_count_stage2(stage1, stage2, val_ds, cfg, label=phase_label)
            except Exception as e:
                print(f"  [per-speaker-count eval] crashed ({type(e).__name__}: {e}); "
                      f"skipping this report, training continues normally")

        if global_step <= len(train_loader):  # i.e. this was the very first epoch of the whole run
            elapsed = time.time() - epoch_t0
            print(f"\n  [budget] first epoch took {elapsed/60:.1f} min. Rough full-run estimate: "
                  f"{elapsed * sum(p['epochs'] for p in cfg['stage1_phases'] + cfg['stage2_phases']) / 3600:.1f} "
                  f"GPU-hours (very rough -- phases have different sizes/costs). "
                  f"Checked against session_time_budget_hours={cfg['session_time_budget_hours']}h per session; "
                  f"will auto-stop cleanly and resume across as many reruns as it takes.\n")


main()



[data] loading train split...
[data] train_mix2: 500 samples (dir=/kaggle/input/datasets/viditarora18/speech-sep-mixtures/train_mix2/kaggle/working/train_mix2, strategy=A (per-sample subfolders))
[data] train_mix3: 500 samples (dir=/kaggle/input/datasets/viditarora18/speech-sep-mixtures/train_mix3/kaggle/working/train_mix3, strategy=A (per-sample subfolders))
[data] train_mix4: 500 samples (dir=/kaggle/input/datasets/viditarora18/speech-sep-mixtures/train_mix4/kaggle/working/train_mix4, strategy=A (per-sample subfolders))
[data] train_mix5: 500 samples (dir=/kaggle/input/datasets/viditarora18/speech-sep-mixtures/train_mix5/kaggle/working/train_mix5, strategy=A (per-sample subfolders))
[data] split='train' TOTAL: 2000 mixtures
[data] loading val split...
[data] dev_mix2: 40 samples (dir=/kaggle/input/datasets/viditarora18/speech-sep-mixtures/dev_mix2/kaggle/working/dev_mix2, strategy=A (per-sample subfolders))
[data] dev_mix3: 40 samples (dir=/kaggle/input/datasets/viditarora18/speech-s